# Gaffer: run the video pipeline on a Colab GPU

Set the runtime first: **Runtime > Change runtime type > GPU** (T4 is fine; L4 / A100 are faster if your plan has them).

Before running, put these in your Google Drive (a folder called `gaffer_colab`):
- the match video (e.g. `match.mp4`)
- `best.pt` : the trained ball model (`.local_data/ball_scale_model/run/weights/best.pt` on the laptop)
- `pitch_mask.npy` : the pitch mask for this camera (`.local_data/aug_pitch_mask.npy` on the laptop), optional

Then run the cells top to bottom. **Stop the runtime when you are done** (Runtime > Disconnect and delete runtime).

In [ ]:
# 1. Is there a GPU?
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# 2. Google Drive (the video and weights live there; results are copied back there)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/gaffer_colab'
!ls -la "$DRIVE"

In [ ]:
# 3. Code. If the repo is private, clone it another way (do NOT paste a password / token into a shared notebook).
%cd /content
!rm -rf gaffer
!git clone --branch experiments/pitch-team-identity-research https://github.com/shayan-sethi/gaffer.git
%cd /content/gaffer

In [ ]:
# 4. Install (a few minutes the first time) and download SAM 2 weights
!pip -q install -r scripts/requirements-pipeline.txt
!SAM2_BUILD_CUDA=0 pip -q install git+https://github.com/facebookresearch/sam2.git
!bash scripts/get_weights.sh

In [ ]:
# 5. Copy the trained ball model and the pitch mask next to the code
!mkdir -p .local_data/ball_scale_model/run/weights
!cp "$DRIVE/best.pt" .local_data/ball_scale_model/run/weights/best.pt
!cp "$DRIVE/pitch_mask.npy" .local_data/pitch_mask.npy 2>/dev/null || echo 'no pitch mask copied'
!ls -la .local_data/ball_scale_model/run/weights

In [ ]:
# 6. Run a clip.  EDIT these:  video name, start second, seconds, pitch band (top y, bottom y)
import time, os
VIDEO = f'{DRIVE}/match.mp4'
START, SECONDS = 2594, 30
BAND_Y0, BAND_Y1 = 560, 1440
MASK = '.local_data/pitch_mask.npy' if os.path.exists('.local_data/pitch_mask.npy') else ''
OUT = '.local_data/colab_out'
t0 = time.time()
!bash scripts/run_clip.sh "$VIDEO" $START $SECONDS $OUT $BAND_Y0 $BAND_Y1 $MASK
print(f'\nTOTAL: {(time.time() - t0) / 60:.1f} minutes for {SECONDS} s of match  ({(time.time() - t0) / SECONDS:.1f}x real time)')

In [ ]:
# 7. Look at the result: passes, teams, and the video (re-encoded so the browser can play it)
!cat $OUT/pass_events.txt
!cat $OUT/teams_report.txt | head -3
!ffmpeg -y -loglevel error -i $OUT/passes_teams.mp4 -vcodec libx264 -pix_fmt yuv420p $OUT/passes_teams_h264.mp4
from IPython.display import Video
Video(f'{OUT}/passes_teams_h264.mp4', embed=True, width=900)

In [ ]:
# 8. Save the results to Drive, then STOP the runtime (Runtime > Disconnect and delete runtime)
!mkdir -p "$DRIVE/results"
!cp -r $OUT "$DRIVE/results/run_$START"
!ls "$DRIVE/results"